# NB11 — Final Official Test Evaluation

**Dissertation:** Explainable and Trustworthy Multimodal Deep Learning for Predictive Maintenance of Industrial Assets  
**Student:** 2023AA05069 | AIMLCZG628T | BITS Pilani  
**Notebook role:** Full training on all 100 FD001 engines followed by one-time official test evaluation  
**Environment:** Google Colab T4  
**Note:** The official test endpoint is opened exactly once in this notebook. No inspection of test results is permitted before training is complete.

---

## Purpose and protocol

NB10 established the repeated-validation result using 80/20 engine-level splits. The best epoch for each neural model was determined from the median across seeds 21, 42 and 84.

This notebook trains each model on the full set of 100 FD001 training engines using those frozen epochs, then evaluates once on the official NASA C-MAPSS FD001 test set. The test labels (`RUL_FD001.txt`) are held out until all training is complete.

**No hyperparameter changes are permitted after viewing test results. This is a one-time endpoint evaluation.**

| Parameter | Value |
|---|---|
| Dataset | NASA C-MAPSS FD001 |
| RUL cap | 125 cycles |
| Rolling window for derived features | 5 cycles |
| Sequence window | 30 cycles |
| Training engines | All 100 FD001 training engines |
| Model seed | 42 (fixed) |
| GRU frozen epochs | 12 |
| DerivedOnlyMLP frozen epochs | 59 |
| MultiViewGRUFusion frozen epochs | 12 |
| Test set access | Once — after all training is complete |

## Section 0 — Colab and Project Setup

I mount Google Drive and set the project root. All paths are relative to `PROJECT_ROOT`.

In [ ]:
import os
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = '/content/drive/MyDrive/dissertation-rul-xai'
else:
    PROJECT_ROOT = os.getcwd()

os.chdir(PROJECT_ROOT)
sys.path.insert(0, PROJECT_ROOT)

RAW_DIR   = os.path.join(PROJECT_ROOT, 'data', 'raw', 'CMAPSS')
PROC_DIR  = os.path.join(PROJECT_ROOT, 'data', 'processed')
MODEL_DIR = os.path.join(PROJECT_ROOT, 'models', 'final')
OUT_DIR   = os.path.join(PROJECT_ROOT, 'reports', 'final_test')

os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(OUT_DIR,   exist_ok=True)

print(f'Project root : {PROJECT_ROOT}')
print(f'Raw data     : {RAW_DIR}')
print(f'Model output : {MODEL_DIR}')
print(f'Report output: {OUT_DIR}')

## Section 1 — Frozen Experiment Configuration

All parameters are copied verbatim from NB10 Section 2. The only additions are:
- `NB11_EPOCHS`: frozen epoch counts derived from NB10 repeated-validation medians
- `FULL_TRAIN_SEED`: model seed for full-training runs (same as MODEL_SEED)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf
import gc
import json
from datetime import datetime, timezone

# ── Dataset parameters ─────────────────────────────────────────────────────
RUL_CAP    = 125
WINDOW_SIZE = 30
ROLLING_WIN = 5
STRIDE      = 1

# ── Feature sets ───────────────────────────────────────────────────────────
SENSOR_COLS = [
    'sensor_measurement_11', 'sensor_measurement_4',  'sensor_measurement_12',
    'sensor_measurement_7',  'sensor_measurement_15', 'sensor_measurement_21',
    'sensor_measurement_20', 'sensor_measurement_2',  'sensor_measurement_17',
    'sensor_measurement_3',  'sensor_measurement_8',  'sensor_measurement_13',
    'sensor_measurement_9',  'sensor_measurement_14',
]
METADATA_COLS = ['unit_number', 'time_in_cycles', 'RUL', 'RUL_capped']
TARGET_COL    = 'RUL_capped'

# ── XGBoost configuration (frozen from NB04) ───────────────────────────────
XGB_PARAMS = dict(
    n_estimators     = 300,
    learning_rate    = 0.05,
    max_depth        = 4,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    objective        = 'reg:squarederror',
    random_state     = 42,
    n_jobs           = -1,
)

# ── Neural model training parameters (frozen from NB10) ───────────────────
GRU_BATCH       = 256
GRU_LR          = 0.001
GRU_LR_FACTOR   = 0.5
GRU_MIN_LR      = 1e-6
GRU_ES_PATIENCE = 10
GRU_LR_PATIENCE = 5

MLP_BATCH       = 128
MLP_LR          = 0.001
MLP_LR_FACTOR   = 0.5
MLP_MIN_LR      = 1e-5
MLP_ES_PATIENCE = 8
MLP_LR_PATIENCE = 4

# ── Frozen epoch selection (medians from NB10 repeated validation) ─────────
NB11_EPOCHS = {
    'GRU':                12,   # median of [12, 12, 26]
    'DerivedOnlyMLP':     59,   # median of [55, 60, 59]
    'MultiViewGRUFusion': 12,   # median of [12, 9, 12]
}

# ── Full-training seed (same as MODEL_SEED throughout) ────────────────────
FULL_TRAIN_SEED = 42

print('Frozen configuration loaded.')
print('NB11 frozen epochs:', NB11_EPOCHS)

## Section 2 — Environment Manifest

In [ ]:
import platform
import subprocess

def get_pkg_version(pkg):
    try:
        import importlib.metadata
        return importlib.metadata.version(pkg)
    except Exception:
        try:
            result = subprocess.run(['pip', 'show', pkg], capture_output=True, text=True)
            for line in result.stdout.splitlines():
                if line.startswith('Version:'):
                    return line.split(':', 1)[1].strip()
        except Exception:
            return 'unknown'

env = {
    'timestamp':   datetime.now(timezone.utc).isoformat(),
    'python':      platform.python_version(),
    'tensorflow':  tf.__version__,
    'numpy':       np.__version__,
    'pandas':      pd.__version__,
    'xgboost':     get_pkg_version('xgboost'),
    'sklearn':     get_pkg_version('scikit-learn'),
    'platform':    platform.platform(),
}
print('Environment:')
for k, v in env.items():
    print(f'  {k}: {v}')

with open(os.path.join(OUT_DIR, 'nb11_experiment_environment.json'), 'w') as f:
    json.dump(env, f, indent=2)

## Section 3 — Shared Preprocessing Functions

Copied verbatim from NB10 Section 4. These are the identical functions used in the repeated-validation experiment.

In [ ]:
import random
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import root_mean_squared_error, mean_absolute_error, r2_score


def compute_derived_features(df, sensor_features, rolling_window=ROLLING_WIN):
    """Per-engine rolling mean, std, delta, and cycle_index. No cross-engine leakage."""
    parts = []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').copy()
        for s in sensor_features:
            unit_df[f'{s}_rmean'] = unit_df[s].rolling(rolling_window, min_periods=1).mean()
            unit_df[f'{s}_rstd']  = unit_df[s].rolling(rolling_window, min_periods=1).std().fillna(0)
            unit_df[f'{s}_delta'] = unit_df[s] - unit_df[s].iloc[0]
        unit_df['cycle_index'] = unit_df['time_in_cycles']
        parts.append(unit_df)
    return pd.concat(parts, ignore_index=True)


def fit_scaler(train_df, feature_set):
    """Fit StandardScaler on training data only."""
    return StandardScaler().fit(train_df[feature_set])


def apply_scaler(df, scaler, feature_set):
    """Apply a pre-fit scaler to a dataframe copy."""
    out = df.copy()
    out[feature_set] = scaler.transform(df[feature_set])
    return out


def create_sequence_windows(df, sensor_cols, target_col, window_size=WINDOW_SIZE, stride=STRIDE):
    """Sliding 30-cycle windows of raw sensor sequence. Windows stay within engine."""
    X, y, meta = [], [], []
    for unit, unit_df in df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            X.append(feats[start:end + 1])
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycles[end],
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return np.array(X, dtype=np.float32), np.array(y, dtype=np.float32), pd.DataFrame(meta)


def create_multiview_windows(b_df, c_df, sensor_cols, derived_cols, target_col,
                              window_size=WINDOW_SIZE, stride=STRIDE):
    """Paired sequence + derived-feature windows, aligned by (unit, cycle)."""
    c_lookup = c_df.set_index(['unit_number', 'time_in_cycles'])
    X_seq, X_der, y, meta = [], [], [], []
    for unit, unit_df in b_df.groupby('unit_number'):
        unit_df = unit_df.sort_values('time_in_cycles').reset_index(drop=True)
        feats   = unit_df[sensor_cols].values.astype(np.float32)
        targets = unit_df[target_col].values.astype(np.float32)
        cycles  = unit_df['time_in_cycles'].values
        raw_rul = unit_df['RUL'].values
        if len(unit_df) < window_size:
            continue
        for end in range(window_size - 1, len(unit_df), stride):
            start = end - window_size + 1
            cycle = cycles[end]
            key   = (unit, cycle)
            if key not in c_lookup.index:
                continue
            X_seq.append(feats[start:end + 1])
            X_der.append(c_lookup.loc[key, derived_cols].values.astype(np.float32))
            y.append(targets[end])
            meta.append({'unit_number': unit, 'time_in_cycles': cycle,
                         'RUL': raw_rul[end], 'RUL_capped': targets[end]})
    return (np.array(X_seq, dtype=np.float32), np.array(X_der, dtype=np.float32),
            np.array(y, dtype=np.float32), pd.DataFrame(meta))


def evaluate_predictions(y_true, y_pred):
    """RMSE, MAE, R² with predictions clipped to [0, RUL_CAP]."""
    y_pred = np.clip(np.asarray(y_pred, dtype=np.float64).reshape(-1), 0, RUL_CAP)
    y_true = np.asarray(y_true, dtype=np.float64).reshape(-1)
    return {
        'rmse': float(root_mean_squared_error(y_true, y_pred)),
        'mae':  float(mean_absolute_error(y_true, y_pred)),
        'r2':   float(r2_score(y_true, y_pred)),
    }


print('Shared preprocessing functions defined.')

## Section 4 — Frozen Model Builders

Copied verbatim from NB10 Section 6. No architectural changes are permitted here.

In [ ]:
from tensorflow.keras import layers, models, callbacks


def build_gru(window_size, n_sensors):
    model = models.Sequential([
        layers.Input(shape=(window_size, n_sensors)),
        layers.GRU(64, return_sequences=True),
        layers.GRU(32),
        layers.Dense(50, activation='relu'),
        layers.Dropout(0.2),
        layers.Dense(1),
    ], name='GRU')
    model.compile(optimizer=tf.keras.optimizers.Adam(GRU_LR), loss='mse', metrics=['mae'])
    return model


def build_derived_mlp(n_features):
    inp = layers.Input(shape=(n_features,), name='degradation_feature_view')
    x   = layers.Dense(64, activation='relu')(inp)
    x   = layers.Dropout(0.2)(x)
    x   = layers.Dense(32, activation='relu')(x)
    out = layers.Dense(1, name='rul_prediction')(x)
    model = models.Model(inputs=inp, outputs=out, name='DerivedOnlyMLP')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


def build_multiview_gru(window_size, n_sensors, n_derived):
    seq_in = layers.Input(shape=(window_size, n_sensors), name='sensor_sequence_view')
    seq_x  = layers.GRU(64, return_sequences=False, name='sensor_gru_encoder')(seq_in)
    seq_x  = layers.Dropout(0.2)(seq_x)

    der_in = layers.Input(shape=(n_derived,), name='degradation_feature_view')
    der_x  = layers.Dense(64, activation='relu', name='degradation_dense_1')(der_in)
    der_x  = layers.Dropout(0.2)(der_x)
    der_x  = layers.Dense(32, activation='relu', name='degradation_dense_2')(der_x)

    fused = layers.Concatenate(name='view_fusion')([seq_x, der_x])
    z     = layers.Dense(64, activation='relu', name='fusion_dense_1')(fused)
    z     = layers.Dropout(0.2)(z)
    z     = layers.Dense(32, activation='relu', name='fusion_dense_2')(z)
    out   = layers.Dense(1, name='rul_prediction')(z)

    model = models.Model(inputs=[seq_in, der_in], outputs=out, name='MultiViewGRUFusion')
    model.compile(optimizer=tf.keras.optimizers.Adam(MLP_LR), loss='mse', metrics=['mae'])
    return model


print('Model builder functions defined.')

## Section 5 — Load Full Training Data

All 100 FD001 training engines are used. There is no train/validation split in this notebook —
the model sees the complete training set for the specified number of epochs.

In [ ]:
INDEX_COLS      = ['unit_number', 'time_in_cycles']
OP_COLS         = [f'operational_setting_{i}' for i in range(1, 4)]
ALL_SENSOR_COLS = [f'sensor_measurement_{i}' for i in range(1, 22)]
ALL_COLS        = INDEX_COLS + OP_COLS + ALL_SENSOR_COLS

raw_train = pd.read_csv(
    os.path.join(RAW_DIR, 'train_FD001.txt'),
    sep=r'\s+', header=None, names=ALL_COLS
)

max_cycles        = raw_train.groupby('unit_number')['time_in_cycles'].max().rename('max_cycle')
raw_train         = raw_train.join(max_cycles, on='unit_number')
raw_train['RUL']  = raw_train['max_cycle'] - raw_train['time_in_cycles']
raw_train['RUL_capped'] = raw_train['RUL'].clip(upper=RUL_CAP).astype(float)
raw_train.drop(columns=['max_cycle'], inplace=True)

all_units = sorted(raw_train['unit_number'].unique())
print(f'Training data loaded: {raw_train.shape}')
print(f'Total training engines: {len(all_units)}')
assert len(all_units) == 100, f'Expected 100 training engines, got {len(all_units)}'
print('Engine count assertion passed.')

## Section 6 — Full-Training Preprocessing

I compute derived features and fit scalers on the complete training set.
These scalers are used both for training and for transforming the test set.

In [ ]:
import joblib

# Compute derived features on full training set
train_with_derived = compute_derived_features(raw_train, SENSOR_COLS, ROLLING_WIN)

derived_cols = (
    [f'{s}_rmean' for s in SENSOR_COLS] +
    [f'{s}_rstd'  for s in SENSOR_COLS] +
    [f'{s}_delta' for s in SENSOR_COLS] +
    ['cycle_index']
)
feature_set_b = SENSOR_COLS        # 14 raw sensors
feature_set_c = derived_cols       # 43 derived features

# Fit scalers on full training set
scaler_b = StandardScaler().fit(train_with_derived[feature_set_b])
scaler_c = StandardScaler().fit(train_with_derived[feature_set_c])

train_b = apply_scaler(train_with_derived, scaler_b, feature_set_b)
train_c = apply_scaler(train_with_derived, scaler_c, feature_set_c)

# Build training arrays
X_train_seq, y_train, train_meta_seq = create_sequence_windows(
    train_b, feature_set_b, TARGET_COL
)
X_train_seq_mv, X_train_der, y_train_mv, train_meta_mv = create_multiview_windows(
    train_b, train_c, feature_set_b, feature_set_c, TARGET_COL
)

n_derived = len(derived_cols)

# Assertions
assert 'normalized_cycle_age' not in feature_set_b + feature_set_c, 'Leakage risk'
assert 'cycle_index' in derived_cols, 'cycle_index missing'
assert X_train_seq.shape == (len(y_train), WINDOW_SIZE, len(feature_set_b))
assert X_train_der.shape[1] == n_derived
assert not np.isnan(X_train_seq).any()
assert not np.isnan(X_train_der).any()

print(f'Derived features: {n_derived}')
print(f'X_train_seq: {X_train_seq.shape}')
print(f'X_train_der: {X_train_der.shape}')
print(f'y_train:     {y_train.shape}')

# Save scalers for test-set transformation
joblib.dump(scaler_b, os.path.join(MODEL_DIR, 'scaler_b_full.joblib'))
joblib.dump(scaler_c, os.path.join(MODEL_DIR, 'scaler_c_full.joblib'))
print('Scalers saved.')

## Section 7 — Full-Training: Frozen Epoch Runs

Each model is trained on all 100 engines for exactly the frozen number of epochs from NB11_EPOCHS.
Early stopping is not used here — training runs for exactly the specified epochs.
The model seed is reset before each model to ensure reproducibility.

In [ ]:
from tensorflow.keras.backend import clear_session
from tensorflow.keras.utils import set_random_seed as set_keras_seed

full_train_results = {}

# ── XGBoost ────────────────────────────────────────────────────────────────
print('=== XGBoost (full training) ===')
np.random.seed(FULL_TRAIN_SEED)
random.seed(FULL_TRAIN_SEED)

# XGBoost uses Feature Set C (derived features), window-aligned rows
# Use the last cycle per engine for XGBoost (no windowing needed)
xgb_train = train_c.copy()
xgb_train['y'] = xgb_train[TARGET_COL]

xgb = XGBRegressor(**XGB_PARAMS)
xgb.fit(xgb_train[feature_set_c], xgb_train['y'])
joblib.dump(xgb, os.path.join(MODEL_DIR, 'XGBoost_C_full.joblib'))
print(f'  XGBoost trained on {len(xgb_train)} rows, saved.')
full_train_results['XGBoost'] = {'model': xgb, 'type': 'xgboost'}

# ── GRU ───────────────────────────────────────────────────────────────────
print('=== GRU (full training) ===')
clear_session(); gc.collect()
set_keras_seed(FULL_TRAIN_SEED)

gru = build_gru(WINDOW_SIZE, len(feature_set_b))
history_gru = gru.fit(
    X_train_seq, y_train,
    epochs     = NB11_EPOCHS['GRU'],
    batch_size = GRU_BATCH,
    verbose    = 1,
)
gru.save(os.path.join(MODEL_DIR, 'GRU_B_full_window30.keras'))
pd.DataFrame(history_gru.history).to_csv(
    os.path.join(OUT_DIR, 'training_history_GRU_full.csv'), index=False
)
print(f'  GRU trained for {NB11_EPOCHS["GRU"]} epochs, saved.')
full_train_results['GRU'] = {'model': gru, 'type': 'keras', 'history': history_gru.history}

# ── DerivedOnlyMLP ────────────────────────────────────────────────────────
print('=== DerivedOnlyMLP (full training) ===')
clear_session(); gc.collect()
set_keras_seed(FULL_TRAIN_SEED)

mlp = build_derived_mlp(n_derived)
history_mlp = mlp.fit(
    X_train_der, y_train_mv,
    epochs     = NB11_EPOCHS['DerivedOnlyMLP'],
    batch_size = MLP_BATCH,
    verbose    = 1,
)
mlp.save(os.path.join(MODEL_DIR, 'DerivedOnlyMLP_full_window30.keras'))
pd.DataFrame(history_mlp.history).to_csv(
    os.path.join(OUT_DIR, 'training_history_DerivedOnlyMLP_full.csv'), index=False
)
print(f'  DerivedOnlyMLP trained for {NB11_EPOCHS["DerivedOnlyMLP"]} epochs, saved.')
full_train_results['DerivedOnlyMLP'] = {'model': mlp, 'type': 'keras', 'history': history_mlp.history}

# ── MultiViewGRUFusion ────────────────────────────────────────────────────
print('=== MultiViewGRUFusion (full training) ===')
clear_session(); gc.collect()
set_keras_seed(FULL_TRAIN_SEED)

fusion = build_multiview_gru(WINDOW_SIZE, len(feature_set_b), n_derived)
history_fusion = fusion.fit(
    [X_train_seq_mv, X_train_der], y_train_mv,
    epochs     = NB11_EPOCHS['MultiViewGRUFusion'],
    batch_size = MLP_BATCH,
    verbose    = 1,
)
fusion.save(os.path.join(MODEL_DIR, 'MultiViewGRUFusion_full_window30.keras'))
pd.DataFrame(history_fusion.history).to_csv(
    os.path.join(OUT_DIR, 'training_history_MultiViewGRUFusion_full.csv'), index=False
)
print(f'  MultiViewGRUFusion trained for {NB11_EPOCHS["MultiViewGRUFusion"]} epochs, saved.')
full_train_results['MultiViewGRUFusion'] = {'model': fusion, 'type': 'keras', 'history': history_fusion.history}

print()
print('All four models trained and saved.')

## Section 8 — Epoch Selection Record

I save the frozen epoch table as an artefact (E12). This documents the NB10 derivation
before any test results are viewed.

In [ ]:
epoch_records = [
    {'model': 'GRU',                'frozen_epochs': 12,
     'nb10_seeds': [12, 12, 26], 'derivation': 'median of seeds 21, 42, 84'},
    {'model': 'DerivedOnlyMLP',     'frozen_epochs': 59,
     'nb10_seeds': [55, 60, 59], 'derivation': 'median of seeds 21, 42, 84'},
    {'model': 'MultiViewGRUFusion', 'frozen_epochs': 12,
     'nb10_seeds': [12,  9, 12], 'derivation': 'median of seeds 21, 42, 84'},
]

epoch_df = pd.DataFrame(epoch_records)
epoch_path = os.path.join(OUT_DIR, 'final_epoch_selection_fd001.csv')
epoch_df.to_csv(epoch_path, index=False)
print('Epoch selection saved:')
print(epoch_df.to_string(index=False))

## Section 9 — Load Official Test Data

The official test set is opened here for the first time. Training is complete and
no model changes are permitted after this point.

The test set contains the last observed cycle for each of 100 test engines.
True RUL labels are in `RUL_FD001.txt`.

In [ ]:
# Load test trajectories
raw_test = pd.read_csv(
    os.path.join(RAW_DIR, 'test_FD001.txt'),
    sep=r'\s+', header=None, names=ALL_COLS
)

# Load true RUL labels
rul_labels = pd.read_csv(
    os.path.join(RAW_DIR, 'RUL_FD001.txt'),
    header=None, names=['true_rul']
)
rul_labels['unit_number'] = range(1, len(rul_labels) + 1)

# The test file contains trajectories up to the last observed cycle.
# True RUL = cycles remaining after the last observed cycle.
# For evaluation, we predict on the last cycle window of each engine.
max_test_cycles = raw_test.groupby('unit_number')['time_in_cycles'].max().rename('max_cycle')
raw_test = raw_test.join(max_test_cycles, on='unit_number')

# Attach true RUL to last cycle of each test engine
last_cycles = raw_test.groupby('unit_number')['time_in_cycles'].max().reset_index()
last_cycles = last_cycles.merge(rul_labels, on='unit_number')
last_cycles['RUL_true_capped'] = last_cycles['true_rul'].clip(upper=RUL_CAP).astype(float)

print(f'Test data loaded:  {raw_test.shape}')
print(f'Test engines:      {raw_test["unit_number"].nunique()}')
print(f'RUL labels loaded: {len(rul_labels)} engines')
print(f'True RUL range (uncapped): [{rul_labels["true_rul"].min()}, {rul_labels["true_rul"].max()}]')

## Section 10 — Test Set Preprocessing

The test set is transformed using the scalers fitted on the full training set.
No new fitting is performed.

In [ ]:
# Compute derived features on test set
test_with_derived = compute_derived_features(raw_test, SENSOR_COLS, ROLLING_WIN)

# Apply training scalers
test_b = apply_scaler(test_with_derived, scaler_b, feature_set_b)
test_c = apply_scaler(test_with_derived, scaler_c, feature_set_c)

# For each test engine, use the last window_size cycles
# (if engine has fewer than window_size cycles, skip — assert none do for FD001)
test_units = sorted(raw_test['unit_number'].unique())
min_test_len = raw_test.groupby('unit_number')['time_in_cycles'].count().min()
print(f'Minimum test engine length: {min_test_len} cycles')
assert min_test_len >= WINDOW_SIZE, (
    f'Test engine with fewer than {WINDOW_SIZE} cycles detected'
)

# Build last-window arrays for each test engine
test_seq_windows  = []   # (100, 30, 14)
test_der_vectors  = []   # (100, 43)
test_units_order  = []

for unit in test_units:
    unit_b = test_b[test_b['unit_number'] == unit].sort_values('time_in_cycles').reset_index(drop=True)
    unit_c = test_c[test_c['unit_number'] == unit].sort_values('time_in_cycles').reset_index(drop=True)

    seq_window = unit_b[feature_set_b].values[-WINDOW_SIZE:].astype(np.float32)
    der_vector = unit_c[feature_set_c].iloc[-1].values.astype(np.float32)

    test_seq_windows.append(seq_window)
    test_der_vectors.append(der_vector)
    test_units_order.append(unit)

X_test_seq = np.array(test_seq_windows, dtype=np.float32)   # (100, 30, 14)
X_test_der = np.array(test_der_vectors, dtype=np.float32)   # (100, 43)

print(f'X_test_seq: {X_test_seq.shape}')
print(f'X_test_der: {X_test_der.shape}')
assert not np.isnan(X_test_seq).any()
assert not np.isnan(X_test_der).any()
print('Test preprocessing complete.')

## Section 11 — Official Test Evaluation

This is the one-time endpoint evaluation. Predictions are made for the last observed cycle
of each test engine and compared against the true RUL labels.

In [ ]:
# Build last-cycle XGBoost feature vectors
xgb_test_rows = []
for unit in test_units_order:
    unit_c = test_c[test_c['unit_number'] == unit].sort_values('time_in_cycles')
    xgb_test_rows.append(unit_c[feature_set_c].iloc[-1].values)
X_test_xgb = np.array(xgb_test_rows, dtype=np.float32)

# True RUL (ordered by test_units_order)
true_rul_ordered = (
    last_cycles.set_index('unit_number')
    .loc[test_units_order, 'true_rul']
    .values.astype(np.float64)
)
true_rul_capped_ordered = np.clip(true_rul_ordered, 0, RUL_CAP)

# ── Predict ───────────────────────────────────────────────────────────────
pred_xgb    = np.clip(xgb.predict(X_test_xgb).astype(np.float64), 0, RUL_CAP)
pred_gru    = np.clip(gru.predict(X_test_seq, verbose=0).reshape(-1).astype(np.float64), 0, RUL_CAP)
pred_mlp    = np.clip(mlp.predict(X_test_der, verbose=0).reshape(-1).astype(np.float64), 0, RUL_CAP)
pred_fusion = np.clip(fusion.predict([X_test_seq, X_test_der], verbose=0).reshape(-1).astype(np.float64), 0, RUL_CAP)

# ── Metrics ───────────────────────────────────────────────────────────────
def test_metrics(y_true, y_pred, model_name):
    rmse = float(root_mean_squared_error(y_true, y_pred))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    me   = float((y_pred - y_true).mean())
    print(f'  {model_name:25s}  RMSE {rmse:7.4f}  MAE {mae:7.4f}  R² {r2:7.4f}  MeanErr {me:+.4f}')
    return {'model': model_name, 'rmse': rmse, 'mae': mae, 'r2': r2, 'mean_error': me}

print('=== OFFICIAL TEST ENDPOINT RESULTS ===')
print()
metrics_rows = [
    test_metrics(true_rul_capped_ordered, pred_xgb,    'XGBoost'),
    test_metrics(true_rul_capped_ordered, pred_gru,    'GRU'),
    test_metrics(true_rul_capped_ordered, pred_mlp,    'DerivedOnlyMLP'),
    test_metrics(true_rul_capped_ordered, pred_fusion, 'MultiViewGRUFusion'),
]

metrics_df = pd.DataFrame(metrics_rows)
metrics_df['dataset'] = 'FD001_test'
metrics_df['evaluation'] = 'official_endpoint'
metrics_df['frozen_epochs_used'] = metrics_df['model'].map(
    lambda m: NB11_EPOCHS.get(m, 'n/a (XGBoost)')
)

metrics_path = os.path.join(OUT_DIR, 'final_test_endpoint_metrics_fd001.csv')
metrics_df.to_csv(metrics_path, index=False)
print()
print(f'Metrics saved to: {metrics_path}')

## Section 12 — Save Endpoint Predictions

I save the per-engine predictions for all four models as a single CSV (E11).

In [ ]:
preds_df = pd.DataFrame({
    'unit_number':          test_units_order,
    'true_rul':             true_rul_ordered,
    'true_rul_capped':      true_rul_capped_ordered,
    'pred_XGBoost':         pred_xgb,
    'pred_GRU':             pred_gru,
    'pred_DerivedOnlyMLP':  pred_mlp,
    'pred_MultiViewGRUFusion': pred_fusion,
})

for col in ['pred_XGBoost', 'pred_GRU', 'pred_DerivedOnlyMLP', 'pred_MultiViewGRUFusion']:
    preds_df[f'error_{col.replace("pred_","")}'] = preds_df[col] - preds_df['true_rul_capped']

preds_path = os.path.join(OUT_DIR, 'final_test_endpoint_predictions_fd001.csv')
preds_df.to_csv(preds_path, index=False)
print(f'Predictions saved: {preds_path}')
print(preds_df.head(10).to_string(index=False))

### NB11 Completion Gate

- [x] All 100 FD001 training engines used for full training
- [x] Model seed reset before each neural model
- [x] GRU trained for exactly 12 epochs (frozen from NB10)
- [x] DerivedOnlyMLP trained for exactly 59 epochs (frozen from NB10)
- [x] MultiViewGRUFusion trained for exactly 12 epochs (frozen from NB10)
- [x] XGBoost trained on full derived feature set
- [x] Test set opened once, after all training complete
- [x] Official endpoint metrics saved (E10)
- [x] Per-engine predictions saved (E11)
- [x] Epoch selection table saved (E12)
- [x] No hyperparameter changes after viewing test results

### NB11 Generated Artefacts

**Reports — `reports/final_test/`**
- `nb11_experiment_environment.json`
- `final_epoch_selection_fd001.csv` — E12
- `final_test_endpoint_metrics_fd001.csv` — E10
- `final_test_endpoint_predictions_fd001.csv` — E11
- `training_history_GRU_full.csv`
- `training_history_DerivedOnlyMLP_full.csv`
- `training_history_MultiViewGRUFusion_full.csv`

**Models — `models/final/`**
- `scaler_b_full.joblib`
- `scaler_c_full.joblib`
- `XGBoost_C_full.joblib`
- `GRU_B_full_window30.keras`
- `DerivedOnlyMLP_full_window30.keras`
- `MultiViewGRUFusion_full_window30.keras`